# Predicción de defectos de software

**Curso:** Inteligencia Artificial  
**Docente:** Julian Rene Muñoz Burbano  
**Coleccion:** 1 · **Proyecto:** 4

El modelo estima P(defecto) por modulo con metricas NASA PC1. El semaforo de riesgo lo aplican reglas de negocio (umbrales, modulo critico, cobertura vs LOC).

In [ ]:
from pathlib import Path
import sys, json
ROOT=Path.cwd().resolve()
if not (ROOT/'config.json').exists():
    ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
config=json.loads((ROOT/'config.json').read_text(encoding='utf-8'))
config

## 1. Dataset y ETL

Fuente: NASA PC1 / OpenML 1068. `descargar_dataset.py` obtiene el CSV y `preparar` limpia metricas McCabe/Halstead.

In [ ]:
from backend.etl import load_processed, FEATURES, TARGET
from pathlib import Path
import subprocess, sys
if not (Path('data')/'raw'/'pc1.csv').exists():
    subprocess.run([sys.executable, 'descargar_dataset.py'], check=True)
df=load_processed()
df[FEATURES+[TARGET]].describe().T.head(8)

## 2. Entrenamiento versionado

Particion estratificada 80/20. LogisticRegression con class_weight=balanced. Artefactos en `models/`.

In [ ]:
from backend.ml import train
info=train(df)
info['metrics']

## 3. Prediccion por modulo (IA + reglas)

In [ ]:
from backend.ml import predict
ejemplo={
    'module_id':'auth-service',
    'metricas':{c:float(df[c].median()) for c in FEATURES},
    'cobertura_tests':0.2,
    'es_modulo_critico':True,
}
ejemplo['metricas']['loc']=260
predict(ejemplo)

## 4. API y dashboard

MongoDB local o `docker compose up`. Terminales: `uvicorn backend.api:app --reload --port 8000` y `streamlit run frontend/app.py`. Documentacion: `/docs`.

Roles: operativo (prediccion), analista (metricas), admin (POST /train).